# Preparation for the QPE demos

Run this notebook once to produce the files `Demo.ipynb` and `Fe2S2_qpe_demo.ipynb` read.
Every step is skipped when its output already exists, so it is safe to re-run.

For each system, in `<molecule>/<nel>e<norb>o/`:

| File | Step |
| --- | --- |
| `<molecule>_avas.gbw` | AVAS start orbitals (Fe2S2 only): RHF, then an AVAS rotation |
| `<molecule>.fcidump`, ORCA output | CASSCF from those orbitals, dumping the active-space integrals |
| `<molecule>_dmrg_info.txt`, `<molecule>_dmrg_circuit.json` | DMRG guide state (simulated systems only) |

The demos only use the two large Fe2S2 active spaces for the resource estimate, which
needs their integrals but not a converged CASSCF. A full ICE-CI CASSCF in those spaces
does not fit a workstation (it needs more than 18 GB for (30e,32o)), so for them Step 2
does a single CASSCF iteration with a loose ICE-CI, enough to dump the integrals in the
AVAS orbitals. Their printed energies are not meaningful.

In [ ]:
# Paths
orca_dir = "~/Software/orca_6_1_1_macosx_arm64_openmpi411"
julia_dir = "~/.juliaup/bin"

# What to prepare: CASSCF + FCIDUMP for all, DMRG only for the simulated systems.
prepare = ["H2 (2e,2o)", "Fe2S2 (10e,10o)", "Fe2S2 (30e,20o)", "Fe2S2 (30e,32o)"]
prepare_dmrg = ["H2 (2e,2o)", "Fe2S2 (10e,10o)"]

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["PATH"] = os.pathsep.join(
    [os.path.expanduser(orca_dir), os.path.expanduser(julia_dir), os.environ["PATH"]]
)

from opi.core import Calculator
from opi.input.blocks import BlockOutput
from opi.input.simple_keywords import BasisSet, Scf, ShellType
from opi.input.structures import Structure

# ORCA runs in parallel through MPI; without mpirun it has to run serially.
orca_cores = os.cpu_count() if shutil.which("mpirun") else 1
print("ORCA:", shutil.which("orca"), f"({orca_cores} core(s))")
print("Julia:", shutil.which("julia"))

## Systems and AVAS active spaces

[Fe<sub>2</sub>S<sub>2</sub>(SCH<sub>3</sub>)<sub>4</sub>]<sup>2&minus;</sup> (charge &minus;2, singlet, def2-SVP).
Atoms are 0-based as in `Fe2S2.xyz`: Fe 0 and 5, bridging S 1 and 4, terminal S 2, 3, 6 and 7.
Each AVAS `system` block selects `norb` orbitals holding `nel` electrons with the most
character of the listed atomic orbitals:

- (10e,10o): Fe 3d.
- (30e,20o): Fe 3d, bridging S 3p, and per terminal S the occupied orbital with the most 3p character.
- (30e,32o): as above, with Fe 4s and 4d added to the Fe block (double shell).

In [ ]:
def avas_block(*groups: str) -> str:
    return "%scf\n avas\n" + "".join(groups) + " end\nend"


FE_3D = """  system
   nel 10
   norb 10
   shell 3, 3, 3, 3, 3, 3, 3, 3, 3, 3
   l     2, 2, 2, 2, 2, 2, 2, 2, 2, 2
   m_l  -2,-1, 0, 1, 2,-2,-1, 0, 1, 2
   center 0, 0, 0, 0, 0, 5, 5, 5, 5, 5
  end
"""
FE_DOUBLE_SHELL = """  system
   nel 10
   norb 22
   shell 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4
   l     2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2
   m_l  -2,-1, 0, 1, 2, 0,-2,-1, 0, 1, 2,-2,-1, 0, 1, 2, 0,-2,-1, 0, 1, 2
   center 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5
  end
"""
S_BRIDGING_3P = """  system
   nel 12
   norb 6
   shell 3, 3, 3, 3, 3, 3
   l     1, 1, 1, 1, 1, 1
   m_l  -1, 0, 1,-1, 0, 1
   center 1, 1, 1, 4, 4, 4
  end
"""
S_TERMINAL_3P = "".join(
    f"""  system
   nel 2
   norb 1
   shell 3, 3, 3
   l     1, 1, 1
   m_l  -1, 0, 1
   center {s}, {s}, {s}
  end
"""
    for s in (2, 3, 6, 7)
)

fe2s2 = dict(molecule="Fe2S2", charge=-2, basis=BasisSet.DEF2_SVP)
# Integrals only: one CASSCF iteration with a loose ICE-CI (see the introduction).
integrals_only = " cistep ice\n maxiter 1\n ci\n  tgen 1e-2\n  maxiter 200\n end\n"
SYSTEMS = {
    "H2 (2e,2o)": dict(molecule="H2", charge=0, basis=BasisSet.STO_3G, nel=2, norb=2, avas=None, casscf=""),
    "Fe2S2 (10e,10o)": dict(**fe2s2, nel=10, norb=10, avas=avas_block(FE_3D), casscf=""),
    "Fe2S2 (30e,20o)": dict(**fe2s2, nel=30, norb=20, avas=avas_block(FE_3D, S_BRIDGING_3P, S_TERMINAL_3P), casscf=integrals_only),
    "Fe2S2 (30e,32o)": dict(**fe2s2, nel=30, norb=32, avas=avas_block(FE_DOUBLE_SHELL, S_BRIDGING_3P, S_TERMINAL_3P), casscf=integrals_only),
}


def workdir(s: dict) -> Path:
    path = Path.cwd() / s["molecule"] / f"{s['nel']}e{s['norb']}o"
    path.mkdir(parents=True, exist_ok=True)
    return path


def calculator(s: dict, basename: str, wd: Path) -> Calculator:
    calc = Calculator(basename=basename, working_dir=wd)
    calc.structure = Structure.from_xyz(Path.cwd() / s["molecule"] / f"{s['molecule']}.xyz")
    calc.structure.charge, calc.structure.multiplicity = s["charge"], 1
    calc.input.ncores = orca_cores
    calc.input.add_simple_keywords(ShellType.RHF, s["basis"])
    return calc


def run(calc: Calculator, label: str) -> None:
    calc.write_input()
    calc.run()
    if not calc.get_output().terminated_normally():
        raise RuntimeError(f"ORCA did not terminate normally: {label}")

## Step 1: AVAS start orbitals

One RHF for Fe2S2 (shared by all active spaces), then one AVAS rotation per active space.
AVAS needs converged orbitals, so it runs on the RHF orbitals without further SCF
iterations (`NoIter`). The rotated orbitals have the active space around the HOMO/LUMO,
so the CASSCF in Step 2 (and in the demos) can start from them directly.

In [ ]:
for name in prepare:
    s = SYSTEMS[name]
    if s["avas"] is None:
        continue
    wd = workdir(s)
    avas_gbw = wd / f"{s['molecule']}_avas.gbw"
    if avas_gbw.exists():
        print(f"{name}: {avas_gbw.name} exists, skipped")
        continue

    rhf_dir = wd.parent / "rhf"
    rhf_dir.mkdir(exist_ok=True)
    rhf_gbw = rhf_dir / f"{s['molecule']}_rhf.gbw"
    if not rhf_gbw.exists():
        print("Running the RHF ...")
        run(calculator(s, f"{s['molecule']}_rhf", rhf_dir), "RHF")

    shutil.copy(rhf_gbw, wd / rhf_gbw.name)
    calc = calculator(s, f"{s['molecule']}_avas", wd)
    calc.input.add_simple_keywords(Scf.MOREAD, Scf.NOITER)
    calc.input.moinp = wd / rhf_gbw.name
    calc.input.add_arbitrary_string(s["avas"])
    run(calc, f"AVAS {name}")
    print(f"{name}: wrote {avas_gbw.name}")

## Step 2: CASSCF and FCIDUMP

CASSCF from the AVAS orbitals, dumping the active-space integrals as `<molecule>.fcidump`.
For H2 and Fe2S2 (10e,10o) this is the same calculation the demos run with
`recompute_orca = True`; for the large spaces it is the integrals-only variant.

In [ ]:
for name in prepare:
    s = SYSTEMS[name]
    wd = workdir(s)
    if (wd / f"{s['molecule']}.fcidump").exists():
        print(f"{name}: FCIDUMP exists, skipped")
        continue

    calc = calculator(s, s["molecule"], wd)
    if s["avas"] is not None:
        calc.input.add_simple_keywords(Scf.MOREAD)
        calc.input.moinp = wd / f"{s['molecule']}_avas.gbw"
    calc.input.memory = 8000  # MB per core; the large ICE-CI needs more than ORCA's default
    calc.input.add_arbitrary_string(f"%casscf\n nel {s['nel']}\n norb {s['norb']}\n mult 1\n{s['casscf']}end")
    dump = BlockOutput()
    dump.add_option("DumpActInts", "True")
    calc.input.add_blocks(dump)
    print(f"{name}: running CASSCF ...")
    run(calc, f"CASSCF {name}")
    output = calc.get_output()
    output.parse()
    print(f"{name}: CASSCF energy = {output.get_final_energy():.10f} Eh")

## Step 3: DMRG guide states

DMRG (MPSCircuits.jl) on the FCIDUMP, compiled into a state-preparation circuit. The
summary (`key=value` lines, including the DMRG energy) is saved next to the circuit.

In [ ]:
for name in prepare_dmrg:
    s = SYSTEMS[name]
    wd = workdir(s)
    circuit_path = wd / f"{s['molecule']}_dmrg_circuit.json"
    info_path = wd / f"{s['molecule']}_dmrg_info.txt"
    if circuit_path.exists() and info_path.exists():
        print(f"{name}: DMRG circuit exists, skipped")
        continue

    print(f"{name}: running DMRG ...")
    out = subprocess.run(
        ["julia", "+1.12", "--project=dmrg_env", "dmrg_circuit.jl",
         str(wd / f"{s['molecule']}.fcidump"), str(circuit_path)],
        capture_output=True, text=True, check=True,
    ).stdout
    info_path.write_text(out)
    print(out)